# Intelligent defence — Baseline + Exhaustive Feature Search

Naive baseline (`from_Intelligent defence` only) vs all 127 combinations of delta TQ.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from itertools import combinations
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Style
BG = '#FAFAFA'; GRID = '#EDEDED'; AXIS = '#D5D5D5'
TEXT = '#2D2D2D'; SUBTEXT = '#737373'
C_BASELINE = '#BF5B3F'; C_TACTICAL = '#2E74B5'
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG,
    'axes.edgecolor': AXIS, 'axes.labelcolor': SUBTEXT,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlepad': 16,
    'axes.labelsize': 9.5, 'axes.grid': False,
    'text.color': TEXT, 'xtick.color': SUBTEXT, 'ytick.color': SUBTEXT,
    'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
    'font.family': 'sans-serif', 'figure.dpi': 140,
    'axes.spines.top': False, 'axes.spines.right': False,
})

In [2]:
DATA_DIR = "../../../../thesis_data/processed_data/thesis_model_dataset/active/"
df = pd.read_parquet(DATA_DIR + "within_league_transfers_v5.parquet")
mf = df[df["from_position"] == "Midfielder"].copy()

QUALITY = 'Intelligent defence'
mf["delta_Q"] = mf["to_" + QUALITY] - mf["from_" + QUALITY]

TEAM_QUALITIES = ["ATTACK", "ATTACKING_TRANSITION", "CHANCE_CREATION",
                  "DEFENCE", "DEFENSIVE_TRANSITION", "OUTCOME", "PENETRATION"]

for tq in TEAM_QUALITIES:
    mf[f"delta_tq_{tq}"] = mf[f"to_q_{tq}"] - mf[f"from_q_proj_{tq}"]

delta_tq = [f"delta_tq_{tq}" for tq in TEAM_QUALITIES]
mf = mf.dropna(subset=["from_" + QUALITY, "delta_Q"] + delta_tq)

train, test = train_test_split(mf, test_size=0.2, random_state=42)
y_train = train["delta_Q"]
y_test  = test["delta_Q"]
print(f"Quality: {QUALITY}")
print(f"Train: {len(train):,}  |  Test: {len(test):,}")

Quality: Intelligent defence
Train: 3,932  |  Test: 984


---
## Naive Baseline

In [3]:
X_tr = sm.add_constant(train[["from_" + QUALITY]])
X_te = sm.add_constant(test[["from_" + QUALITY]])
naive = sm.OLS(y_train, X_tr).fit()
naive_pred = naive.predict(X_te)
naive_r2 = r2_score(y_test, naive_pred)
naive_mae = mean_absolute_error(y_test, naive_pred)
print(f"Naive — R²: {naive_r2:.4f}  |  MAE: {naive_mae:.4f}")

Naive — R²: 0.1904  |  MAE: 0.4733


---
## Exhaustive Search (127 combinations)

In [4]:
results = []

for k in range(1, len(delta_tq) + 1):
    for combo in combinations(delta_tq, k):
        feat = ["from_" + QUALITY] + list(combo)
        X_tr = sm.add_constant(train[feat])
        X_te = sm.add_constant(test[feat])
        model = sm.OLS(y_train, X_tr).fit()
        pred = model.predict(X_te)
        results.append({
            "n_deltas": k,
            "deltas": ", ".join(c.replace("delta_tq_", "") for c in combo),
            "R2_test": r2_score(y_test, pred),
            "MAE_test": mean_absolute_error(y_test, pred),
            "R2_adj_train": model.rsquared_adj,
            "BIC": model.bic,
        })

results_df = pd.DataFrame(results).sort_values('R2_test', ascending=False)
print(f"Total combinations: {len(results_df)}")

Total combinations: 127


---
## Top 10 by Test R²

In [5]:
print(f"Naive baseline R²: {naive_r2:.4f}")
print()
print(results_df.head(10).to_string(index=False, float_format="{:.4f}".format))

Naive baseline R²: 0.1904

 n_deltas                                              deltas  R2_test  MAE_test  R2_adj_train       BIC
        2                                DEFENCE, PENETRATION   0.1939    0.4728        0.2433 7109.5243
        3          DEFENCE, DEFENSIVE_TRANSITION, PENETRATION   0.1938    0.4728        0.2431 7117.6829
        1                                             DEFENCE   0.1938    0.4729        0.2435 7101.2668
        2                       DEFENCE, DEFENSIVE_TRANSITION   0.1937    0.4729        0.2433 7109.4260
        3                       DEFENCE, OUTCOME, PENETRATION   0.1933    0.4729        0.2452 7106.6893
        4 DEFENCE, DEFENSIVE_TRANSITION, OUTCOME, PENETRATION   0.1933    0.4730        0.2451 7114.8439
        2                                    DEFENCE, OUTCOME   0.1931    0.4731        0.2454 7098.5071
        3                        ATTACK, DEFENCE, PENETRATION   0.1930    0.4729        0.2432 7117.1426
        3              DEFEN

---
## Best per Number of Deltas

In [6]:
best_per_k = results_df.loc[results_df.groupby('n_deltas')['R2_test'].idxmax()]
best_per_k = best_per_k.sort_values('n_deltas')
print(f"Naive baseline R²: {naive_r2:.4f}")
print()
print(best_per_k.to_string(index=False, float_format="{:.4f}".format))

Naive baseline R²: 0.1904

 n_deltas                                                                                             deltas  R2_test  MAE_test  R2_adj_train       BIC
        1                                                                                            DEFENCE   0.1938    0.4729        0.2435 7101.2668
        2                                                                               DEFENCE, PENETRATION   0.1939    0.4728        0.2433 7109.5243
        3                                                         DEFENCE, DEFENSIVE_TRANSITION, PENETRATION   0.1938    0.4728        0.2431 7117.6829
        4                                                DEFENCE, DEFENSIVE_TRANSITION, OUTCOME, PENETRATION   0.1933    0.4730        0.2451 7114.8439
        5                          ATTACKING_TRANSITION, DEFENCE, DEFENSIVE_TRANSITION, OUTCOME, PENETRATION   0.1925    0.4736        0.2457 7118.7299
        6                  ATTACK, ATTACKING_TRANSITION, DEFE

In [7]:
best = results_df.iloc[0]
best_deltas_str = best['deltas']
best_r2 = best['R2_test']
best_mae = best['MAE_test']
print('Best combo: ' + best_deltas_str)
print('R2 test: {:.4f}  |  MAE: {:.4f}'.format(best_r2, best_mae))
print('R2 gain over naive: {:+.4f}'.format(best_r2 - naive_r2))
print()

best_deltas = ['delta_tq_' + d.strip() for d in best_deltas_str.split(',')]
best_feat = ['from_' + QUALITY] + best_deltas
X_tr = sm.add_constant(train[best_feat])
best_model = sm.OLS(y_train, X_tr).fit()
print(best_model.summary())

Best combo: DEFENCE, PENETRATION
R2 test: 0.1939  |  MAE: 0.4728
R2 gain over naive: +0.0035

                            OLS Regression Results                            
Dep. Variable:                delta_Q   R-squared:                       0.244
Model:                            OLS   Adj. R-squared:                  0.243
Method:                 Least Squares   F-statistic:                     422.3
Date:                Fri, 20 Mar 2026   Prob (F-statistic):          9.20e-238
Time:                        15:20:39   Log-Likelihood:                -3538.2
No. Observations:                3932   AIC:                             7084.
Df Residuals:                    3928   BIC:                             7110.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------